# Supervised Text Classification with Logistic Regression
### IMDB Movie Review Sentiment Analysis

This notebook continues from `eda.ipynb`, which explored the IMDB dataset and produced
a cleaned file `imdb_cleaned.csv` (columns: `final_review`, `label`).

Here we build **supervised classification models** using **Logistic Regression** to
predict review sentiment (positive = 1, negative = 0). Two feature representations are
compared — **Bag-of-Words (CountVectorizer)** and **TF-IDF** — since the choice of
feature extraction strongly affects a text classifier's performance.

**Workflow**
1. Load the cleaned data
2. Train/test split
3. Feature extraction (BoW and TF-IDF)
4. Train Logistic Regression on each feature set
5. Evaluate and compare both models
6. Interpret the model (most influential words)
7. Tune hyperparameters with GridSearchCV
8. Save the final model for reuse


## 1. Imports and Data Loading

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)
import pickle

sns.set_style("whitegrid")


In [ ]:
# Load the cleaned dataset produced by eda.ipynb
# (columns: 'final_review' - preprocessed text, 'label' - 1 = positive, 0 = negative)
df = pd.read_csv("imdb_cleaned.csv")

# Drop any rows where cleaning left an empty string, to avoid issues during vectorisation
df = df.dropna(subset=["final_review", "label"])
df = df[df["final_review"].str.strip() != ""]

print("Dataset shape:", df.shape)
df[["final_review", "label"]].head()


## 2. Train/Test Split

We hold out 20% of the data for testing, stratified by label so both classes stay balanced in each split.

In [ ]:
X = df["final_review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keep the positive/negative ratio consistent across train and test
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")


## 3. Feature Extraction

Logistic Regression needs numeric input, so raw text must be converted into vectors.
We build **two** feature sets to compare later:

- **Bag-of-Words (CountVectorizer):** counts how often each word/n-gram appears
- **TF-IDF (TfidfVectorizer):** down-weights words that are common across most reviews,
  emphasising words that are more distinctive to a given review

Both are limited to the top 5000 features and include unigrams + bigrams, to keep
training fast while still capturing short phrases (e.g. "not good").

In [ ]:
# --- Bag-of-Words features ---
bow_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# --- TF-IDF features ---
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("BoW feature matrix shape:  ", X_train_bow.shape)
print("TF-IDF feature matrix shape:", X_train_tfidf.shape)


## 4. Train Logistic Regression Models

In [ ]:
# Model 1: Logistic Regression on Bag-of-Words features
lr_bow = LogisticRegression(max_iter=1000, random_state=42)
lr_bow.fit(X_train_bow, y_train)

# Model 2: Logistic Regression on TF-IDF features
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)

print("Both models trained successfully.")


## 5. Evaluation

We define a small helper so both models are scored the same way.

In [ ]:
def evaluate_model(model, X_test_features, y_test, model_name):
    """Print key classification metrics and return them as a dict for later comparison."""
    y_pred = model.predict(X_test_features)
    y_proba = model.predict_proba(X_test_features)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"--- {model_name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

    return {"model": model_name, "accuracy": acc, "precision": prec,
            "recall": rec, "f1": f1, "roc_auc": auc, "y_pred": y_pred, "y_proba": y_proba}


results_bow = evaluate_model(lr_bow, X_test_bow, y_test, "Logistic Regression (Bag-of-Words)")
results_tfidf = evaluate_model(lr_tfidf, X_test_tfidf, y_test, "Logistic Regression (TF-IDF)")


### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, res, name in zip(
    axes,
    [results_bow, results_tfidf],
    ["Bag-of-Words", "TF-IDF"]
):
    cm = confusion_matrix(y_test, res["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negative", "positive"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix - {name}")

plt.tight_layout()
plt.show()


### ROC Curves

In [ ]:
plt.figure(figsize=(6, 6))

for res, name in zip([results_bow, results_tfidf], ["Bag-of-Words", "TF-IDF"]):
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## 6. Model Comparison

In [ ]:
# Collect metrics into a single comparison table
comparison_df = pd.DataFrame([
    {k: v for k, v in results_bow.items() if k not in ("y_pred", "y_proba")},
    {k: v for k, v in results_tfidf.items() if k not in ("y_pred", "y_proba")},
]).set_index("model")

comparison_df


In [ ]:
# Visualise the comparison
comparison_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot(
    kind="bar", figsize=(10, 5), rot=0
)
plt.title("Logistic Regression Performance: Bag-of-Words vs TF-IDF")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.legend(loc="lower right")
plt.show()


## 7. Model Interpretation

Logistic Regression coefficients are directly interpretable: a large **positive**
coefficient pushes the prediction toward "positive" sentiment, and a large **negative**
coefficient pushes it toward "negative" sentiment. We inspect the TF-IDF model here since
TF-IDF weighting generally gives cleaner, more distinctive top words.

In [ ]:
# Pair each feature (word/bigram) with its learned coefficient
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
coefficients = lr_tfidf.coef_[0]

top_n = 15
top_positive_idx = np.argsort(coefficients)[-top_n:]
top_negative_idx = np.argsort(coefficients)[:top_n]

top_positive = pd.DataFrame({
    "word": feature_names[top_positive_idx],
    "coefficient": coefficients[top_positive_idx]
}).sort_values("coefficient", ascending=True)

top_negative = pd.DataFrame({
    "word": feature_names[top_negative_idx],
    "coefficient": coefficients[top_negative_idx]
}).sort_values("coefficient", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].barh(top_negative["word"], top_negative["coefficient"], color="firebrick")
axes[0].set_title("Top Words Driving 'Negative' Predictions")
axes[0].set_xlabel("Coefficient")

axes[1].barh(top_positive["word"], top_positive["coefficient"], color="seagreen")
axes[1].set_title("Top Words Driving 'Positive' Predictions")
axes[1].set_xlabel("Coefficient")

plt.tight_layout()
plt.show()


## 8. Hyperparameter Tuning

The regularisation strength `C` controls the trade-off between fitting the training
data closely and keeping the model simple (smaller `C` = stronger regularisation).
We use `GridSearchCV` with 5-fold cross-validation on the TF-IDF features, the
stronger-performing representation, to find a better value of `C`.

In [ ]:
param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train_tfidf, y_train)

print("Best C:", grid_search.best_params_["C"])
print("Best cross-validated F1-score:", round(grid_search.best_score_, 4))

# Evaluate the tuned model on the held-out test set
best_lr = grid_search.best_estimator_
results_tuned = evaluate_model(best_lr, X_test_tfidf, y_test, "Logistic Regression (TF-IDF, Tuned)")


## 9. Save the Final Model

We persist the tuned model together with its TF-IDF vectorizer so both can be reloaded and reused for inference without retraining.

In [ ]:
with open("logistic_regression_sentiment_model.pkl", "wb") as f:
    pickle.dump(best_lr, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

print("Saved model to 'logistic_regression_sentiment_model.pkl'")
print("Saved vectorizer to 'tfidf_vectorizer.pkl'")


## 10. Quick Inference Check

A small sanity check that loads the saved artefacts and predicts sentiment for a couple
of new, unseen sentences.

In [ ]:
def predict_sentiment(text, model, vectorizer):
    """Predict sentiment (and confidence) for a single raw text string."""
    features = vectorizer.transform([text])
    pred = model.predict(features)[0]
    proba = model.predict_proba(features)[0][pred]
    label = "positive" if pred == 1 else "negative"
    return label, proba


sample_reviews = [
    "this movie was absolutely wonderful and the acting was superb",
    "what a boring and terrible waste of time"
]

for review in sample_reviews:
    label, confidence = predict_sentiment(review, best_lr, tfidf_vectorizer)
    print(f"Review: {review!r}")
    print(f"Predicted sentiment: {label} (confidence: {confidence:.2%})\n")
